In [1]:
#https://adamheins.com/blog/lagrangian-mechanics-three-ways#5
import numpy as np
import math
from scipy.integrate import odeint
import sympy as sym
from sympy import diff
from sympy.matrices import Matrix
from sympy.physics.mechanics import dynamicsymbols, init_vprinting
import matplotlib.pyplot as plt

In [2]:
# link masses
M1 = 3
M2 = 2
M3 = 1

# link lengths
LX = 0.1
LY = 0.1
L2 = 1
L3 = 0.5

# moments of inertia (base link doesn't rotate so don't need it's inertia)
I2 = M2 * L2 ** 2 / 12
I3 = M3 * L3 ** 2 / 12

# inertia matrices
G1 = np.diag(np.array([M1, M1, 0]))
G2 = np.diag(np.array([M2, M2, I2]))
G3 = np.diag(np.array([M3, M3, I3]))

# gravity
G = 9.8

In [3]:
# joint configuration and velocity are functions of time
t = sym.symbols("t")
q = sym.Array([sym.Function("x1")(t), sym.Function("θ2")(t), sym.Function("θ3")(t)])
dq = q.diff(t)
dq

[Derivative(x1(t), t), Derivative(θ2(t), t), Derivative(θ3(t), t)]

In [8]:
P1 = sym.Matrix([q[0], 0, 0])
P2 = P1 + sym.Matrix([
        LX + 0.5 * L2 * sym.cos(q[1]),
        LY + 0.5 * L2 * sym.sin(q[1]),
        q[1],
    ])
P3 = P2 + sym.Matrix([
        0.5 * L2 * sym.cos(q[1]) + 0.5 * L3 * sym.cos(q[1] + q[2]),
        0.5 * L2 * sym.sin(q[1]) + 0.5 * L3 * sym.sin(q[1] + q[2]),
        q[2],
    ])

In [9]:
P1

Matrix([
[x1(t)],
[    0],
[    0]])

In [10]:
P2

Matrix([
[x1(t) + 0.5*cos(θ2(t)) + 0.1],
[        0.5*sin(θ2(t)) + 0.1],
[                       θ2(t)]])

In [11]:
P3

Matrix([
[x1(t) + 0.25*cos(θ2(t) + θ3(t)) + 1.0*cos(θ2(t)) + 0.1],
[        0.25*sin(θ2(t) + θ3(t)) + 1.0*sin(θ2(t)) + 0.1],
[                                         θ2(t) + θ3(t)]])

In [16]:
J1 = P1.jacobian(q)
J2 = P2.jacobian(q)
J3 = P3.jacobian(q)


In [19]:
J3

Matrix([
[1, -0.25*sin(θ2(t) + θ3(t)) - 1.0*sin(θ2(t)), -0.25*sin(θ2(t) + θ3(t))],
[0,  0.25*cos(θ2(t) + θ3(t)) + 1.0*cos(θ2(t)),  0.25*cos(θ2(t) + θ3(t))],
[0,                                         1,                        1]])

In [20]:
    # mass matrix
M = J1.transpose() * G1 * J1 + J2.transpose() * G2 * J2 + J3.transpose() * G3 * J3
M

Matrix([
[                                      6.0,                                                                                                                      -0.25*sin(θ2(t) + θ3(t)) - 2.0*sin(θ2(t)),                                                                                                                                       -0.25*sin(θ2(t) + θ3(t))],
[-0.25*sin(θ2(t) + θ3(t)) - 2.0*sin(θ2(t)),                (-0.25*sin(θ2(t) + θ3(t)) - 1.0*sin(θ2(t)))**2 + (0.25*cos(θ2(t) + θ3(t)) + 1.0*cos(θ2(t)))**2 + 0.5*sin(θ2(t))**2 + 0.5*cos(θ2(t))**2 + 0.1875, -0.25*(-0.25*sin(θ2(t) + θ3(t)) - 1.0*sin(θ2(t)))*sin(θ2(t) + θ3(t)) + 0.25*(0.25*cos(θ2(t) + θ3(t)) + 1.0*cos(θ2(t)))*cos(θ2(t) + θ3(t)) + 0.0208333333333333],
[                 -0.25*sin(θ2(t) + θ3(t)), -0.25*(-0.25*sin(θ2(t) + θ3(t)) - 1.0*sin(θ2(t)))*sin(θ2(t) + θ3(t)) + 0.25*(0.25*cos(θ2(t) + θ3(t)) + 1.0*cos(θ2(t)))*cos(θ2(t) + θ3(t)) + 0.0208333333333333,                                                          

In [24]:
mass_matrix = sym.lambdify([q], M)
def force(cls, q, dq, ddq):
        M = cls.mass_matrix(q)
        #C = cls.coriolis_matrix(q, dq)
        #g = cls.gravity_vector(q)

        return M 

In [ ]:
force(cls, q, dq, ddq)